# Re-ask once on an invalid contract (rq1), optionally after the data pre-check (pc1rq1) (GPU)

`eval/ext_reask.py` reads an existing parser run (only read), sends **only** the flagged tasks back to the same model
once ("Your contract is invalid: <error> ..." or, with the pre-check, "Your contract does not fit the input data:
<issues> ..."), and writes a new parse folder `<system>_rq1` / `<system>_pc1rq1`. One extra model call per re-asked task.
For the 7B p1-v3 run that is 18 tasks (rq1) or 20 (pc1rq1).

After the run: download `reask_bundle.zip`, unzip it at the project root (folders land in
`results/stage1/extensions/`), then `.venv/bin/python eval/ext_score.py`.
Fine-tuned parsers: add their parse folder (in the repo under `kaggle_outputs/parsing/`) and their adapter path
(attach the adapter zip as a Kaggle dataset).

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
BACKEND = "hf"                  # "hf" | "vllm" | "gold" (pipeline check only, NOT results)
# (parse folder under kaggle_outputs/parsing/, model that wrote it, 4-bit, adapter path or None)
RUNS = [
    ("7b_p1-v3", "Qwen/Qwen2.5-7B-Instruct", True, None),
    # ("3b_p1-v2", "Qwen/Qwen2.5-3B-Instruct", False, None),
    # ("1_5b_p1-v3", "Qwen/Qwen2.5-1.5B-Instruct", False, None),
    # ("ft_1_5b", "Qwen/Qwen2.5-1.5B-Instruct", True, "/kaggle/input/<your-adapter-dataset>/adapter"),
]
MODES = ["rq1", "pc1rq1"]       # rq1 = invalid contracts only; pc1rq1 = + valid contracts that fail the pre-check
OUT_ROOT = "/kaggle/working/results/stage1/extensions"
CACHE = "/kaggle/working/llm_cache"
NEEDED = ["eval/ext_reask.py", "src/igt/agents/reask.py", "src/igt/contracts/precheck.py"] + \
         [f"kaggle_outputs/parsing/{r[0]}/parses.jsonl" for r in RUNS]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox / verifier / tools (own folder) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
if BACKEND == "vllm":
    subprocess.run(["pip", "install", "-q", "-r", "requirements-vllm.txt"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers; print('pandas', pandas.__version__, '| numpy', numpy.__version__, "
         "'| torch', torch.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
def sh(cmd, log=None):
    """Run a step, stream its output live, keep the full log."""
    t0 = time.time()
    p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {' '.join(cmd)}\n" + "".join(lines) + f"\n--- exit {rc}, {(time.time() - t0) / 60:.1f} min ---\n")
    assert rc == 0, f"failed: {' '.join(cmd)}"
    return (time.time() - t0) / 60

for folder, model, q, adapter in RUNS:
    for mode in MODES:
        cmd = ["python", "eval/ext_reask.py", folder, "--backend", BACKEND, "--out-root", OUT_ROOT, "--cache-dir", CACHE]
        cmd += ["--model", model] if BACKEND != "gold" else []
        cmd += ["--load-in-4bit"] if q and BACKEND != "gold" else []
        cmd += ["--adapter", adapter] if adapter else []
        cmd += ["--precheck"] if mode == "pc1rq1" else []
        print(f"== {folder} {mode}: {sh(cmd, f'{OUT_ROOT}/logs/{folder}_{mode}.log'):.1f} min")
for d in sorted(os.listdir(OUT_ROOT)):
    if os.path.exists(f"{OUT_ROOT}/{d}/env.json"):
        e = json.load(open(f"{OUT_ROOT}/{d}/env.json"))
        open(f"{OUT_ROOT}/{d}/git_commit.txt", "w").write(COMMIT)
        print(f"{d}: re-asked {e['n_reasked']}, valid {e['valid_before']} -> {e['valid_after']}")

In [ ]:
# ---------------- ONE zip with everything needed on the laptop (unzip at the project root) ----------------
BUNDLE = f"{WORK}/reask_bundle.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["/kaggle/working/results/stage1/extensions"]:
        for dp, _, fs in os.walk(base):
            for f in fs:
                full = os.path.join(dp, f)
                z.write(full, os.path.relpath(full, WORK))
    z.writestr("reask_bundle_info.json", json.dumps(dict(commit=COMMIT, created=datetime.datetime.now().isoformat(timespec="seconds")), indent=2))
print("download:", BUNDLE, f"({os.path.getsize(BUNDLE) / 1e6:.1f} MB)")